In this article, we are going to stress test the Disjoint Support Autoencoder.   

The algorithm works well enough for clean synthetic data, with global noise added from $\sigma=0.01$.  

There are a few hyperparameters for the algorithm which also require testing. We will do that here.  

In the tests I've done, these are the observable aspects of different performance of the algorithm. I'll add them one by one.  
For now, we are creating a test set while observing aspects of the model which help us with the hyperparameters.  

- does the sigma_0 and sigma_enc actually end up being the sigmas of weights in the model after training?  
  - do for simple L2 regularised.
  - do for disjoint with log
  - do for disjoint without log

<!-- - Decoder initialisation
  - If the decoder weights are initialised using SVD, we get pretty determistic results across seed runs. For each test case, we will also look at the SVD initialisation separately (along with many runs of randomly intialised weights)
- Baseline performance v/s disjoint support performance
- Number of dead atoms
  - Note that sometimes, we end up dead atoms even when `n_components` is less than the ideal number of components.  
- Disjoint support metric: off-diagonal elements in the Gram matrix should be vanishingly close to 0.   
- Seed stability metric
  - We use the hungarian matching algorithm   
  - In summary, we take two atom sets created from separate seeds
    - find cosine similarity matrix
    - solve the linear sum problem to make pairs of atoms which are closest to each other
    - take the similarity mean
  - This is done for every pair of atom sets. The whole thing is again averaged.   -->


A single test works on a specific dataset with some specific corruptions if any. Some examples:

- Normal pure synthetic data as baseline test.  
- Add gaussian noise to each sample for some ratio of samples.  
- Corrupt a fixed atom for some ratio of samples.  

The tests are structured with the test type at the top. Then we look at the properties of the algorithm in these test types.  


# Gradients of our loss terms

First checking the gradient on W only, the disjoint loss only affects it.  

$$
\frac{\partial \mathcal{L}}{\partial W_{ij}} = 
\frac{-2(x_i - \hat{x}_i)s_j}{\sigma_\epsilon^2} 
+ \frac{2W_{ij}\,\phi(w,i-1,j)}{\sigma_0^2} 
- 2\alpha W_{ij} \sum_{r > i} \frac{1}{\phi(w,r-1,j)} 
+ \frac{2\alpha W_{ij}}{\sigma_0^2} \sum_{r > i} W_{rj}^2
$$

$$
\phi(w, i, j)=1 + \alpha\sum_{k=0}^{i}w_{k j}^2
$$

Expectation of MSE gradient, assuming at the beggining, $W=\mathcal{N}(0,\beta_0)$, $S=\mathcal{N}(0,\beta_s)$, $X=\mathcal{N}(0,\sigma_x)$ ($X$ is always this, 0 centered).   

$$
E\left[\left(\frac{\partial MSE}{\partial W_{ij}}\right)^2\right] = 4\frac{\sigma_x^2 \beta_s^2 + \beta_w^2 \beta_s^4 (C+2)}{\sigma_{\epsilon}^2}
$$

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.utils import *
from pt_to_api.utils import (
    show_single_channel_red_green_black as S,
    to_show_list as tsl,
)
from pt_to_api import disjoint_ae, disjoint_ae_learned_sig
from sklearn.metrics.pairwise import (
    pairwise_distances,
    cosine_similarity,
    cosine_distances,
)
from scipy.optimize import linear_sum_assignment
from collections import defaultdict
import numpy as np
from torch import nn
from torch import optim
import warnings
from dataclasses import dataclass
from typing import Any
import math
import gc
import pandas as pd


MODE = "light"

In [ ]:
class MeanPerDimGlobalStdScaler:
    def __init__(self):
        self.means_ = None
        self.global_std_ = None

    def fit(self, X):
        X = np.array(X)
        self.means_ = X.mean(axis=0)
        self.global_std_ = X.std()
        return self

    def transform(self, X):
        X = np.array(X)
        return (X - self.means_) / self.global_std_

    def inverse_transform(self, X):
        X = np.array(X)
        return (X * self.global_std_) + self.means_

In [ ]:
@dataclass
class SingleRun:
    model: Any
    codes: np.ndarray
    components: np.ndarray
    recon: np.ndarray
    loss: float
    hyperparameters: dict

In [ ]:
def make_dim_partition(patch_dim, n_components, seed=42):
    rng = np.random.RandomState(seed)
    perm = rng.permutation(patch_dim)
    return [list(perm[i::n_components]) for i in range(n_components)]

def generate_synthetic_patches(
    patch_dim=72,
    n_components=10,
    k=3,
    n_samples=1000,
    noise_std=0.01,
    seed=42,
    sigma_x=1,
):
    rng = np.random.RandomState(seed)

    dim_partition = make_dim_partition(patch_dim, n_components, seed)

    # ground truth atoms, nonzero only on owned dims
    W_true = np.zeros((n_components, patch_dim))
    for i, dims in enumerate(dim_partition):
        W_true[i, dims] = rng.randn(len(dims))
    W_true /= np.linalg.norm(W_true, axis=1, keepdims=True)

    # each sample uses at most k atoms
    codes_true = np.zeros((n_samples, n_components))
    for i in range(n_samples):
        k_i = rng.randint(1, k + 1)  # active atoms: 1..k
        idx = rng.choice(n_components, k_i, replace=False)
        codes_true[i, idx] = rng.randn(k_i)

    X = codes_true @ W_true

    scale = sigma_x / X.std()
    X *= scale
    W_true *= scale  # keeps codes_true @ W_true ≈ X
    X += rng.randn(*X.shape) * noise_std

    return X, W_true, codes_true, dim_partition

def generate_nonstationary_data(
    n_samples=1000,
    input_dim=20,
    low_variance=0.1,
    high_variance=10.0,
    random_state=None,
):
    rng = np.random.default_rng(random_state)

    n_low = n_samples // 2
    n_high = n_samples - n_low

    X_low = rng.normal(scale=low_variance**0.5, size=(n_low, input_dim))
    X_high = rng.normal(scale=high_variance**0.5, size=(n_high, input_dim))

    X = np.concatenate([X_low, X_high], axis=0)
    return X

def generate_dominant_pc_data(
    n_samples=1000,
    input_dim=9,
    dominant_variance=10.0,
    noise_variance=0.1,
    random_state=None,
):
    rng = np.random.default_rng(random_state)

    # one strong direction
    dominant_direction = rng.normal(size=input_dim)
    dominant_direction /= np.linalg.norm(dominant_direction)

    # scores along that direction
    dominant_scores = rng.normal(scale=dominant_variance**0.5, size=n_samples)

    # small isotropic noise
    noise = rng.normal(scale=noise_variance**0.5, size=(n_samples, input_dim))

    X = dominant_scores[:, None] * dominant_direction[None, :] + noise
    return X

def show_closest_component_of_W_for_each_component(components, W_true, figsize=(5, 2)):
    """
    Given two arrays of numpy vectors of same shapes
    for every component in `components`, this function shows the array in `W_true`
    which has the maximum cosine similarity with the component
    """
    sims = np.abs(cosine_similarity(components, W_true))
    pairs = []
    for i in range(len(components)):
        j = np.argmax(sims[i])
        pairs.append((i, j, sims[i][j]))
    for i, j, score in pairs:
        S(
            [components[i].reshape(3, 3), W_true[j].reshape(3, 3)],
            figsize,
            mode=MODE,
            suptitle=f"similarity score={score}",
            ax_titles=["component", "ground_truth"],
            viztype="local",
        )
        plt.show()


def evaluate_recovery(W_learned, W_true, threshold=0.95):
    """
    W_learned: (n_atoms, patch_dim)
    W_true: (n_atoms, patch_dim)

    for each true atom, finds the best matching learned atom by cosine similarity
    returns fraction of true atoms recovered above threshold
    """
    W_l = W_learned / (np.linalg.norm(W_learned, axis=1, keepdims=True) + 1e-8)
    W_t = W_true / (np.linalg.norm(W_true, axis=1, keepdims=True) + 1e-8)

    sim = np.abs(W_l @ W_t.T)  # (n_atoms, n_atoms), abs because sign is arbitrary
    best_match = sim.max(
        axis=0
    )  # for each true atom, best cosine with any learned atom

    recovered = (best_match >= threshold).mean()
    print(f"Mean best cosine similarity: {best_match.mean():.4f}")
    print(f"Fraction recovered (>{threshold}): {recovered:.4f}")
    return best_match, recovered


def match_atoms(D1: np.ndarray, D2: np.ndarray):
    """
    Match atoms of D1 to atoms of D2 using the Hungarian algorithm
    on cosine distances. Assumes square dictionaries (same n_components).

    D1, D2: shape (n_components, n_features) — sklearn's components_ layout.

    Returns:
        row_ind, col_ind: matched index arrays
        matched_similarities: per-pair cosine similarities
        mean_sim: mean cosine similarity across matched pairs
    """
    # Guard against dead atoms (zero-norm rows produce NaN cosine distances)
    norms_1 = np.linalg.norm(D1, axis=1, keepdims=True)
    norms_2 = np.linalg.norm(D2, axis=1, keepdims=True)
    if np.any(norms_1 == 0) or np.any(norms_2 == 0):
        raise ValueError(
            "One or more atoms have zero norm. "
            "Remove or replace dead atoms before matching."
        )

    # cost = cosine_distances(D1, D2)          # shape (n_components, n_components), values in [0, 2]
    cost = np.abs(cosine_similarity(D1, D2))
    cost = 1 - cost

    row_ind, col_ind = linear_sum_assignment(cost)
    matched_similarities = 1 - cost[row_ind, col_ind]
    mean_sim = float(matched_similarities.mean())
    return row_ind, col_ind, matched_similarities, mean_sim


def get_live(components):
    dead = find_dead_atoms(components).numpy()
    live = np.array([i for i in range(components.shape[0]) if i not in dead])
    return components[live]


def hungarian_match(all_components: list[np.ndarray]):
    """
    Pairwise similarity matching across runs using the Hungarian algorithm.

    Args:
        all_components: list of arrays, each shape (n_components, n_features).
                        All arrays must have the same shape.

    Returns:
        upper: 1-D array of pairwise similarities for all unique pairs
        stability_score: mean of upper
        best_run_idx: index of the run most similar to all others
        pairwise_sims: (n_runs, n_runs) symmetric similarity matrix, diagonal = 1
    """
    n_runs = len(all_components)

    if n_runs < 2:
        raise ValueError("Need at least 2 runs to compute pairwise similarity.")

    shapes = [d.shape for d in all_components]
    if len(set(shapes)) != 1:
        raise ValueError(f"All dictionaries must have the same shape. Got: {shapes}")

    pairwise_sims = np.ones((n_runs, n_runs))
    for i in range(n_runs):
        for j in range(i + 1, n_runs):
            _, _, _, mean_sim = match_atoms(all_components[i], all_components[j])
            pairwise_sims[i, j] = mean_sim
            pairwise_sims[j, i] = mean_sim

    upper = pairwise_sims[np.triu_indices(n_runs, k=1)]
    stability_score = float(upper.mean())

    # Exclude self-similarity (diagonal=1) when ranking runs
    np.fill_diagonal(pairwise_sims, 0)
    mean_sim_per_run = pairwise_sims.sum(axis=1) / (n_runs - 1)
    best_run_idx = int(np.argmax(mean_sim_per_run))
    np.fill_diagonal(pairwise_sims, 1)  # restore diagonal

    return upper, stability_score, best_run_idx, pairwise_sims


def find_dead_atoms(W, threshold=0.1):
    if not isinstance(W, torch.Tensor):
        W = torch.tensor(W)

    peak = W.abs().max(dim=1).values
    peak_normalised = peak / peak.max()

    return torch.where(peak_normalised < threshold)[0]

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim, n_components):
        super().__init__()
        self.encoder = nn.Linear(input_dim, n_components, bias=True)
        self.decoder = nn.Linear(n_components, input_dim, bias=False)

    def forward(self, x):
        codes = self.encoder(x)
        recon = self.decoder(codes)
        return recon, codes


def recon_loss(x, recons, sigma_eps):
    """Reconstruction loss. MSE"""
    return gauss_loss(x, recons) / (sigma_eps * sigma_eps)


def codes_loss(codes, sigma_s):
    """L2 loss on encoder"""
    return gauss_loss(codes, 0) / (sigma_s * sigma_s)


def gauss_loss(x, mean):
    loss = (x - mean) ** 2
    return torch.sum(loss, 1).mean()


def weights_loss(alpha, sigma_0, W):
    """Vectorized version the Weight loss"""
    W_sq = W**2  # (C, K)
    cumsum = torch.cumsum(W_sq, dim=1)  # (C, K), cumsum[c,k] = sum W[c,0..k]^2
    phi = alpha * torch.roll(cumsum, 1, dims=1) + 1  # (C, K)
    phi[:, 0] = 1  # k=0: phi_weight(W, c, -1, alpha) = alpha*0 + 1
    comp1 = (W_sq * phi) / (sigma_0 * sigma_0)
    comp2 = -torch.log(phi)
    return comp1, comp2


def get_scaled_hyperparamters(
    sigma_x,
    input_dim,
    n_components,
    eps_ratio=100,
    w_to_eps_ratio=5,
    alpha_constant=5000.0,
    sigma_s_rel_to_0="equal",
):
    """
    sigma_x      : std of your data
    eps_ratio    : sigma_x / sigma_eps (default 100)
    w_to_eps_ratio: sigma_0 / sigma_eps (default 5)
    alpha_constant: the c in alpha = c / sigma_0^2
    """
    # print("sigma_s_rel_to_0", sigma_s_rel_to_0)
    sigma_eps = sigma_x / eps_ratio
    # make them equal
    if sigma_s_rel_to_0 == "equal":
        sigma_0 = sigma_x / np.sqrt(n_components)
        sigma_s = sigma_x / np.sqrt(n_components)
    elif sigma_s_rel_to_0 == "less":
        sigma_s = sigma_eps * w_to_eps_ratio
        sigma_0 = sigma_x / (
            np.sqrt(n_components) * sigma_s
        )
    else:
        sigma_0 = sigma_eps * w_to_eps_ratio
        sigma_s = sigma_x / (
            np.sqrt(n_components) * sigma_0
        )


    sigma_enc = sigma_s / (
        np.sqrt(input_dim) * sigma_x
    )  # from D*sigma_enc^2*sigma_x^2 = sigma_s^2
    alpha = alpha_constant / (sigma_0**2)

    return dict(
        sigma_eps=sigma_eps,
        sigma_0=sigma_0,
        sigma_s=sigma_s,
        sigma_enc=sigma_enc,
        alpha=alpha,
    )


def init_encoder_using_normal(model, sigma_enc):
    # encoder weights
    nn.init.normal_(model.encoder.weight, mean=0.0, std=sigma_enc)
    nn.init.zeros_(model.encoder.bias)


def init_model_parameters_using_normal(model, scaled_hyperparameters):
    p = scaled_hyperparameters
    sigma_0, sigma_enc = p["sigma_0"], p["sigma_enc"]

    # decoder = W, init with sigma_0
    nn.init.normal_(model.decoder.weight, mean=0.0, std=sigma_0)
    init_encoder_using_normal(model, sigma_enc)


def init_model_parameters_using_svd(model, X, n_components, scaled_hyperparameters):
    p = scaled_hyperparameters

    # svd
    _, _, Vt = np.linalg.svd(X, full_matrices=False)
    w = torch.tensor(Vt[:n_components].T, dtype=torch.float32)
    # scale
    w = (w / w.std()) * p["sigma_0"]
    model.decoder.weight.data = w

    # encoder weights
    init_encoder_using_normal(model, p["sigma_enc"])


def get_hyperparameters_and_init(
    model, X, n_components, input_dim, svd_init=False, eps_ratio=100, sigma_s_rel_to_0="equal"
):
    sigma_x = X.std()
    scaled_hyperparameters = get_scaled_hyperparamters(
        sigma_x,
        input_dim,
        n_components,
        eps_ratio=eps_ratio,
        w_to_eps_ratio=5,
        alpha_constant=5000,
        sigma_s_rel_to_0=sigma_s_rel_to_0,
    )
    if svd_init:
        init_model_parameters_using_svd(model, X, n_components, scaled_hyperparameters)
    else:
        init_model_parameters_using_normal(model, scaled_hyperparameters)
    return scaled_hyperparameters


def train(
    X,
    n_components,
    lr=1e-3,
    epochs=2000,
    batch_size=256,
    verbose=True,
    svd_init=False,
    initialised_model=None,
    use_ln_term=True,
    sigma_s_rel_to_0="equal",
) -> SingleRun:
    """
    X: numpy array (n_samples, input_dim)
    n_components: number of dictionary atoms
    alpha: lorentzian sigma shrinker parameter
    """
    print("use ln term", use_ln_term)
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape

    if initialised_model is None:
        model = Autoencoder(input_dim, n_components)
    else:
        model = initialised_model

    scaled_hyperparameters = get_hyperparameters_and_init(
        model, X, n_components, input_dim, svd_init, 100, sigma_s_rel_to_0
    )

    p = scaled_hyperparameters
    sigma_eps, sigma_0, sigma_s, _, alpha = (
        p["sigma_eps"],
        p["sigma_0"],
        p["sigma_s"],
        p["sigma_enc"],
        p["alpha"],
    )

    losses = defaultdict(list)

    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        # shuffle
        idx = torch.randperm(n_samples)
        permuted_X_t = X_t[idx]

        for i in range(0, n_samples, batch_size):
            batch = permuted_X_t[i : i + batch_size]
            recon, codes = model(batch)

            _recon_loss = recon_loss(batch, recon, sigma_eps)
            _codes_loss = codes_loss(codes, sigma_s)
            comp1, comp2 = weights_loss(alpha, sigma_0, model.decoder.weight)
            comp1, comp2 = comp1.sum(), comp2.sum()

            # weight_loss = comp1 + comp2
            if use_ln_term:
                weight_loss = comp1 + comp2
            else:
                weight_loss = comp1

            losses["comp1"].append(comp1.item())
            losses["comp2"].append(comp2.item())
            losses["recon"].append(_recon_loss.item())
            losses["codes"].append(_codes_loss.item())

            loss = _recon_loss + weight_loss + _codes_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if verbose and epoch % 200 == 0:
            print(
                f"epoch {epoch:4d} | recon_loss {_recon_loss:.4f} weight_loss {weight_loss.sum():.4f} codes_loss {_codes_loss:.4f}"
            )

    with torch.no_grad():
        recon, codes = model(X_t)

    return SingleRun(
        model,
        codes.numpy(),
        model.decoder.weight.T.detach().numpy(),
        recon.numpy(),
        ((X_t - recon) ** 2).sum(1).mean(),
        scaled_hyperparameters,
    )


def train_baseline(
    X,
    n_components,
    lr=1e-3,
    epochs=2000,
    batch_size=256,
    verbose=True,
    eps_ratio=100,
):
    """Train the autoencoder with just reconstruction loss, to find an arbitrary linear model which fits the data

    The main training code requires sigma_eps
    the standard deviation of expected gaussian noise
    when the curve is fitted using Y=WX
    We can generally do a simple sweep of hyperparams
    or use simple heuristics
    If the data is linearly "fittable",
    then we get a good starting point
    using this function.
    """
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape

    model = Autoencoder(input_dim, n_components)

    scaled_hyperparameters = get_hyperparameters_and_init(
        model, X, n_components, input_dim, False, eps_ratio
    )
    p = scaled_hyperparameters
    print("using hyperparameters")
    for k, v in p.items():
        print(f"{k}: {v.item():.4f}")
    sigma_eps, sigma_0, sigma_s = p["sigma_eps"], p["sigma_0"], p["sigma_s"]

    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        idx = torch.randperm(n_samples)
        permuted_X_t = X_t[idx]
        for i in range(0, n_samples, batch_size):
            batch = permuted_X_t[i : i + batch_size]
            recon, codes = model(batch)
            _recon_loss = recon_loss(batch, recon, sigma_eps)
            _codes_loss = codes_loss(codes, sigma_s)
            _weights_loss = gauss_loss(model.decoder.weight, 0) / (sigma_0 * sigma_0)

            loss = _recon_loss + _codes_loss + _weights_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if verbose and epoch % 200 == 0:
            print(f"finetune epoch {epoch:4d} | recon_loss {_recon_loss:.4f}")

    with torch.no_grad():
        recon, codes = model(X_t)

    return SingleRun(
        model,
        codes.numpy(),
        model.decoder.weight.T.detach().numpy(),
        recon.numpy(),
        ((X_t - recon) ** 2).sum(1).mean(),
        scaled_hyperparameters,
    )

In [ ]:
# from typing import Any
# from dataclasses import dataclass


# @dataclass
# class SingleRun:
#     codes: np.ndarray
#     components: np.ndarray
#     recon: np.ndarray
#     X: np.ndarray
#     baseline_loss: float
#     loss: float
#     gram_error: float
#     dead_atom_count: int
#     model: Any


# def plot_runs(runs: list[SingleRun]):
#     fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

#     losses = [r.loss for r in runs]
#     baselines = [r.baseline_loss for r in runs]
#     gram_errors = [r.gram_error for r in runs]
#     dead_counts = [r.dead_atom_count for r in runs]

#     axes[0].plot(losses, color="red", label="loss")
#     axes[0].plot(baselines, color="blue", label="baseline loss")
#     axes[0].legend()
#     axes[0].set_title("Loss")

#     axes[1].plot(gram_errors, color="green")
#     axes[1].set_title("Gram Error")

#     axes[2].plot(dead_counts, color="orange")
#     axes[2].set_title("Dead Atoms")

#     axes[2].set_xlabel("Run")
#     plt.tight_layout()
#     plt.show()


# def mse(x, recon):
#     return ((x - recon) ** 2).sum()



In [ ]:
def std_ratios(v1, v2):
    rat = min(v1, v2) / max(v1, v2)
    if isinstance(rat, torch.Tensor):
        return rat.item()
    return rat

def get_metrics_from_run(run: SingleRun, W_true):
    _, _, sim_vector, mean_sim = match_atoms(W_true, run.components)
    return {
        "mse": run.loss,
        "gram": gram_orthogonality_error(run.components.T),
        "decoder_sigma_ratio": std_ratios(
            run.model.decoder.weight.std(), run.hyperparameters["sigma_0"]
        ),
        "encoder_sigma_ratio": std_ratios(
            run.model.encoder.weight.std(), run.hyperparameters["sigma_enc"]
        ),
        "mean_sim": mean_sim,
        "vec_sim": sim_vector,
    }

def print_summary(X, W_true, codes_true, run: SingleRun):
    print("standard devications")
    print("\tX:", X.std())
    print("\tW_true:", W_true.std())
    print("\tcodes_true:", codes_true.std())
    print("\tcodes:", run.codes.std(), "sigma_s", run.hyperparameters["sigma_s"])
    print("\tcomponents:", run.components.std())
    print("\trecon:", run.recon.std())
    print("\tdecoder:", run.model.decoder.weight.std().item(), "sigma_0:", run.hyperparameters["sigma_0"])
    print("\tencoder:", run.model.encoder.weight.std().item(), "sigma_enc:", run.hyperparameters["sigma_enc"])
    print("\tMSE:", (X - run.recon).std(), "sigma_eps:", run.hyperparameters["sigma_eps"])

# Pure dataset without any noise

This is a kind of baseline.  
We don't even add any noise. Every input is a pure reconstruction using a linear combination of $k$ atoms.  

In [ ]:
N_COMPONENTS = 3

In [ ]:
X, W_true, codes_true, dim_partition = generate_synthetic_patches(
    patch_dim=9,
    n_components=N_COMPONENTS,
    k=3,
    noise_std=0,
)
ws_to_show = [w.reshape(3, 3) for w in W_true]
S(
    ws_to_show,
    (8, 2),
    ncols=N_COMPONENTS,
    mode=MODE,
    suptitle="ground truth basis vectors \n[bright green = high positive, bright red = high negative, white = near zero]\nA vector is of size 1x9, but is shown as 3x3 for easier visibility",
)
plt.show()
S(
    [X[0].reshape(3, 3)] + ws_to_show,
    (8, 2),
    N_COMPONENTS + 1,
    suptitle="First input, with its basis components, the coefficient of each component is it's title",
    mode=MODE,
    ax_titles=["X"] + [f"{codes_true[1][k]:.4f}" for k in range(N_COMPONENTS)],
)
plt.show()
S(
    [X[1].reshape(3, 3)] + ws_to_show,
    (8, 2),
    N_COMPONENTS + 1,
    suptitle="Second input",
    mode=MODE,
    ax_titles=["X"] + [f"{codes_true[1][k]:.4f}" for k in range(N_COMPONENTS)],
)
plt.show()


# Relation between $\sigma_s$ and $\sigma_0$

We can either set $\sigma_s$ to $c\sigma_{\epsilon}$ and derive $\sigma_0$ from it. or the other way round.  
This will determine which one is bigger.  

The relationship we use is simply $R\sigma_0^2\sigma_s^2 = \sigma_x^2$ ($R$ is the number of atoms, or rows in components).  

- We can either set one of them relative to $sigma_eps$ and derive the other
- We can make them equal, ie. $\sigma_0 = \sigma_s = \frac{\sigma_x}{\sqrt{R}}$

Below we will empirically see that setting them to equal is a good condition, for smaller examples. The test will continue for bigger examples later.  


We test for dimensions = `[10, 100]` with `[20%, 50%, 80%]` active atoms.   

In [ ]:
# no noise, pure combination of W_true
X, W_true, codes_true, dim_partition = generate_synthetic_patches(
    patch_dim=9,
    n_components=N_COMPONENTS,
    k=3,
    noise_std=0,
    sigma_x=1,
)

scaler = MeanPerDimGlobalStdScaler().fit(X)
X_scaled = scaler.transform(X)

In [ ]:
# there is some reasonable "it works lol" kinda thing going here
run = train_baseline(X, N_COMPONENTS)
# model, codes, components, recon, scaled_hyperparamters, losses = run
# the hyperparameters match here
print_summary(X, W_true, codes_true, run)

I need simple things to test on first.   


Things that change:
- number of components
- number of dims
- number of components active at a time

The simplest is checking how many atoms we can recover, we keep all the components active to keep it simple. For a given dimension.  
We will assume that data = 100 * components-number.  

In [ ]:
X, W_true, codes_true, dim_partition = generate_synthetic_patches(1000, 200, 200, 200*100, 0.1)

In [ ]:
dims_list = [10, 100]
# for each dim, we test for 20%, 50%, 80% atoms for now, to understand how i can read the data
atoms_ratio = [0.2, 0.5, 0.8]

noise_std = 0.1

runs = {}
inputs = {}

for dim in dims_list:
    for a in atoms_ratio:
        for sigma_s_rel_to_0 in ["equal", "less", "greater"]:
            print("#######", dim, a)
            gc.collect()
            atoms = math.ceil(dim*a)
            # keep all active
            k = atoms
            n_samples = 100*atoms
            X, W_true, codes_true, dim_partition = generate_synthetic_patches(dim, atoms, k, n_samples, noise_std)
    
            scaler = MeanPerDimGlobalStdScaler().fit(X)
            X_scaled = scaler.transform(X)
            run = train(X_scaled, atoms, 1e-3, 4000, 256, True, False, None, True, sigma_s_rel_to_0)
    
            runs[(dim, a, sigma_s_rel_to_0)] = run
            inputs[(dim, a, sigma_s_rel_to_0)] = (X, W_true, codes_true, dim_partition)

## Results

If you look at the table, you'll see that the sigma_s_to_0 gives mean simimilary (the column `mean_sim`) = 99%.  

Surprisingly, we see that if we set $\sigma_s < \sigma_0$, `dims=10` gives 55% mean similarity. The reason is the corresponding MSE. Its much higher than the equal case, we are not able to reconstruct well, so every atom is just a disjoint noise.  

For `dims=100`, $\sigma_s > \sigma_0$, we see 30-60% similarity. The MSE is fine too. On closer inspection, you'll see that the disjoint loss has driven all atoms to near 0. If you look at individual components of a single reconstruction, you'll see random atoms lighting up. This is because the codes have too much variance and are basically taking care of getting the reconstruction.  

It's hard to say why this happens. It's best to ignore that. For now, $\sigma_s = \sigma_0$ gives the best results.  
It also has the good property to simply remove the relation with $\sigma_eps$, which we can use to freely guide reconstruction.  

The next experiments will use the "equal" method

In [ ]:
mets = []
for k in runs:
    r, i = runs[k], inputs[k]
    m = get_metrics_from_run(r, i[1])
    m["dims"] = k[0]
    m["atom_ratio"] = k[1]
    m["sigma_s_to_0"] = k[2]
    mets.append(m)

df = pd.DataFrame(mets)
df["mse"] = df['mse'].apply(lambda x: x.item())
df = df.drop(columns=["vec_sim"])

# df.sort_values("sigma_s_to_0")

In [ ]:
# all equal values
df[df["sigma_s_to_0"] == "equal"]

In [ ]:
# all less
df[df["sigma_s_to_0"] == "less"]

In [ ]:
# all greater
df[df["sigma_s_to_0"] == "greater"]

# log term usefulness

Closely inspecting the gradients generally tells us that the log term has very small gradients through out.  
The log term is basically pushing in the opposite direction from disjointness. It wants to keep weights "non-zero".  
The signal is very low though compared to the actual disjoint term. And we anyways rely on correct reconstruction to give us non-zero weights.  

This section checks if descent becomes easier in the synthetic case if we give up the log term. It is useful to only check the results in the previous section with the `sigma_s_to_0 = equal` case.  


In [ ]:
dims_list = [10, 100]
# for each dim, we test for 20%, 50%, 80% atoms for now, to understand how i can read the data
atoms_ratio = [0.2, 0.5, 0.8]

noise_std = 0.1

runs = {}
inputs = {}

for dim in dims_list:
    for a in atoms_ratio:
        for use_ln in [True, False]:
            print("#######", dim, a)
            gc.collect()
            atoms = math.ceil(dim*a)
            # keep all active
            k = atoms
            n_samples = 100*atoms
            X, W_true, codes_true, dim_partition = generate_synthetic_patches(dim, atoms, k, n_samples, noise_std)
    
            scaler = MeanPerDimGlobalStdScaler().fit(X)
            X_scaled = scaler.transform(X)
            run = train(X_scaled, atoms, 1e-3, 4000, 256, True, False, None, use_ln, "equal")
    
            runs[(dim, a, use_ln)] = run
            inputs[(dim, a, use_ln)] = (X, W_true, codes_true, dim_partition)

In [ ]:
mets = []
for k in runs:
    r, i = runs[k], inputs[k]
    m = get_metrics_from_run(r, i[1])
    m["dims"] = k[0]
    m["atom_ratio"] = k[1]
    m["use_ln"] = k[2]
    mets.append(m)

df = pd.DataFrame(mets)
df["mse"] = df['mse'].apply(lambda x: x.item())
df = df.drop(columns=["vec_sim"])

df

## Results

We don't see a lot of difference. Although removing the `ln` component did bring down `mean_sim` in one case (row 3).  
It has also degraded the MSE in some cases, so this is not very conclusive.  

We will need to test this on other datasets to see if there is problem with the component. Otherwise we go ahead while staying faithful to the probabilistic model.  

# How much sparsity can we take?

The probabilistic treatment exclusively assumes Gaussian distribution on weights and coefficients. This might give us bad results for sparse data.  
We'll also check with and without log in this case, as the `log` term doesn't like sparsity.  


Sparsity can be introduced in multiple ways:

- sparsity within atoms (atmost `k` dims alive in an atom)
- sparsity within codes (atmost `k` codes alive in a sample)
- sparsity across the whole sample (at most $k$ dims are alive in a given sample)
  - this is quite hard
  - we'll simply do this by combining the first two things to make a sparse dataset.

In [ ]:
def generate_synthetic_patches_sparse_atoms(
    patch_dim=72,
    n_components=10,
    k=3,
    k_atom=3,
    n_samples=1000,
    noise_std=0.01,
    seed=42,
    sigma_x=1,
):
    rng = np.random.RandomState(seed)
    dim_partition = make_dim_partition(patch_dim, n_components, seed)

    W_true = np.zeros((n_components, patch_dim))
    for i, dims in enumerate(dim_partition):
        n_active = rng.randint(1, min(k_atom, len(dims)) + 1)
        active_dims = rng.choice(dims, n_active, replace=False)
        W_true[i, active_dims] = rng.randn(len(active_dims))

    W_true /= np.linalg.norm(W_true, axis=1, keepdims=True)

    codes_true = np.zeros((n_samples, n_components))
    for i in range(n_samples):
        k_i = rng.randint(1, k + 1)
        idx = rng.choice(n_components, k_i, replace=False)
        codes_true[i, idx] = rng.randn(k_i)

    X = codes_true @ W_true
    scale = sigma_x / X.std()
    X *= scale
    W_true *= scale
    X += rng.randn(*X.shape) * noise_std

    return X, W_true, codes_true, dim_partition

In [ ]:
# atmost 2 dims active in an atom
X, W_true, codes_true, dim_partition = generate_synthetic_patches_sparse_atoms(25, 3, 3, 2, seed=None)

S([c.reshape(5,5) for c in W_true], 5, len(W_true), mode=MODE)

In [ ]:
import math

dims_list = [10, 100]
# for each dim, we test for 20%, 50%, 80% atoms for now, to understand how i can read the data
atoms_ratio = [0.2, 0.5, 0.8]
# 5%, 20%
# we have already tested with 100%, so going higher might not be productive
sparsity_ratio = [0.05, 0.2, 0.5]

noise_std = 0.1

runs = {}
inputs = {}

for dim in dims_list:
    for a in atoms_ratio:
        for sp_rat in sparsity_ratio:
            print("#######", dim, a)
            gc.collect()
            atoms = math.ceil(dim*a)
            # keep all active
            k = atoms
            n_samples = 100*atoms
            active_dims = math.ceil(dim*sp_rat)

            
            X, W_true, codes_true, dim_partition = generate_synthetic_patches_sparse_atoms(dim, atoms, k, active_dims)
    
            scaler = MeanPerDimGlobalStdScaler().fit(X)
            X_scaled = scaler.transform(X)
            run = train(X_scaled, atoms, 1e-3, 4000, 256, True, False, None, True, "equal")
    
            runs[(dim, a, sp_rat)] = run
            inputs[(dim, a, sp_rat)] = (X, W_true, codes_true, dim_partition)

In [ ]:
mets = []
for k in runs:
    r, i = runs[k], inputs[k]
    m = get_metrics_from_run(r, i[1])
    m["dims"] = k[0]
    m["atom_ratio"] = k[1]
    m["use_ln"] = k[2]
    mets.append(m)

df = pd.DataFrame(mets)
df["mse"] = df['mse'].apply(lambda x: x.item())
df = df.drop(columns=["vec_sim"])

df

In [ ]:
# 80 comps, 100 dims
k = (100, 0.8, 0.5)
rind, cind, sim_vec, mean_sim = match_atoms(runs[k].components, inputs[k][1])

In [ ]:
# we see some weird dims here
# so this is fine i guess
idx = 0
for idx in range(0,5):
    S([runs[k].components[rind[idx]].reshape(10,10), inputs[k][1][cind[idx]].reshape(10,10)], mode=MODE)
    plt.show()

In [ ]:
import math

dims_list = [1000, 10_000]
# for each dim, we test for 20%, 50%, 80% atoms for now, to understand how i can read the data
atoms_ratio = [0.2, 0.5]
# 5%, 20%
# we have already tested with 100%, so going higher might not be productive
sparsity_ratio = [0.05, 0.2]

noise_std = 0.1

runs = {}
inputs = {}

for dim in dims_list:
    for a in atoms_ratio:
        for sp_rat in sparsity_ratio:
            print("#######", dim, a)
            gc.collect()
            atoms = math.ceil(dim*a)
            # keep all active
            k = atoms
            n_samples = 100*atoms
            active_dims = math.ceil(dim*sp_rat)

            
            X, W_true, codes_true, dim_partition = generate_synthetic_patches_sparse_atoms(dim, atoms, k, active_dims)
    
            scaler = MeanPerDimGlobalStdScaler().fit(X)
            X_scaled = scaler.transform(X)
            run = train(X_scaled, atoms, 1e-3, 4000, 256, True, False, None, True, "equal")
    
            runs[(dim, a, sp_rat)] = run
            inputs[(dim, a, sp_rat)] = (X, W_true, codes_true, dim_partition)

## Results

Its not bad, sigma stays the same, we need to increase sparsity i guess. MSE is fine, mean similarity is also fine.  
We'll test on more sparse data